# HYPERVIEW2 Pretraining Transfer Check

Notebook do sprawdzenia hipotezy: czy checkpoint Mamby trenowany na HySpecNet-11k jest lepszym punktem startowym do adaptacji HYPERVIEW2 niz trening od zera.

Porownujemy:

1. `scratch`: trening Mamby od zera na HYPERVIEW2,
2. `pretrained_full`: HySpecNet pretrained -> fine-tune wszystkich wag,
3. `pretrained_frozen_spectral`: HySpecNet pretrained -> zamrozony spectral backbone, uczone tylko `spatial_condition`, `encoder_to_latent`, `entropy_bottleneck`, `decoder`.

Kontrole downstream:

- `original`,
- `spectral_resample_passthrough`: `230 -> HySpecNet-202 -> 230`, bez modelu,
- rekonstrukcje kazdego checkpointu.

Wyniki sa diagnostyka transferu HYPERVIEW2/PRISMA, nie reference-comparable wynikami HySpecNet-11k.

## 1. Ustawienia

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_HV2_ROOT = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DRIVE_RECONS = DRIVE_HSI / 'reconstructions/hyperview2_pretraining_transfer'
DRIVE_RESULTS = DRIVE_HSI / 'downstream_results/hyperview2_pretraining_transfer'
DRIVE_RUNS = DRIVE_HSI / 'runs/hyperview2_pretraining_transfer'

CONFIG_REL = 'configs/mamba/hyperview2_prisma_hyspecnet202_mamba_k4_spatial_rd_lambda_0_001_spectral_feature_ft.yaml'
PRETRAINED_CKPT_CANDIDATES = [
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_spectral_feature_ft_best.pt',
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_ft_best.pt',
    DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_01_best.pt',
]

# quick: szybka odpowiedz czy trend ma sens; balanced/full sa do finalnych tabel.
EVAL_PRESET = 'quick'  # quick | balanced | full
RUN_MODES = ['scratch', 'pretrained_full', 'pretrained_frozen_spectral']
FRACTIONS_BY_PRESET = {
    'quick': [0.05],
    'balanced': [0.01, 0.05, 0.10],
    'full': [0.01, 0.05, 0.10, 1.00],
}
EPOCHS_BY_PRESET = {'quick': 8, 'balanced': 20, 'full': 40}
TRAIN_VAL_SUBSET_BY_PRESET = {'quick': 150, 'balanced': 250, 'full': None}
DOWNSTREAM_TRAIN_SUBSET_BY_PRESET = {'quick': 600, 'balanced': 1000, 'full': None}
DOWNSTREAM_VAL_SUBSET_BY_PRESET = {'quick': 150, 'balanced': 250, 'full': None}
REGRESSORS_BY_PRESET = {
    'quick': ['extra_trees'],
    'balanced': ['extra_trees', 'random_forest'],
    'full': ['hist_gradient_boosting', 'extra_trees', 'random_forest'],
}

RUN_TRAINING = True
RUN_RECONSTRUCTION = True
RUN_DOWNSTREAM = True
SKIP_COMPLETED_TRAINING = True
RECOMPUTE_RECONSTRUCTIONS = False
RECOMPUTE_REGRESSORS = True
REQUIRE_CUDA = True
DISABLE_WANDB = True
RESUME = False
FORCE_REINSTALL_ENV = False

SEED = 42
VAL_FRACTION = 0.2
RECON_BATCH_SIZE = 1
# Colab/Jupyter can emit multiprocessing cleanup assertions with DataLoader workers > 0.
# Keep notebook-side loaders single-process; tensor compute still runs on GPU.
COLAB_DATALOADER_NUM_WORKERS = 0
RECON_NUM_WORKERS = COLAB_DATALOADER_NUM_WORKERS
FEATURE_BATCH_SIZE = 128
FEATURE_NUM_WORKERS = COLAB_DATALOADER_NUM_WORKERS
EXPERIMENT_PREFIX = 'hyperview2_pretrain_transfer'

print('Preset:', EVAL_PRESET)
print('Fractions:', FRACTIONS_BY_PRESET[EVAL_PRESET])
print('Modes:', RUN_MODES)

## 2. Repo

In [ ]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

for module_name in list(sys.modules):
    if module_name == 'hsi_compression' or module_name.startswith('hsi_compression.'):
        del sys.modules[module_name]

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
print('Git:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 3. Zaleznosci

Komorka instaluje Torch 2.7.1 CUDA 12.6 oraz prebuilt wheels `causal-conv1d` / `mamba-ssm`. Po pierwszej instalacji runtime zostanie zrestartowany. Po reconnect uruchom notebook ponownie od komorki repo; marker instalacji pominie reinstall.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, '-m', 'pip']
ENV_MARKER = Path('/content/.hsi_compression_hv2_pretraining_env_v1_torch27_mamba232')


def run(cmd, *, required=True):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = f'Command failed with exit code {result.returncode}: {" ".join(map(str, cmd))}'
        if required:
            raise RuntimeError(message)
        print('Optional command failed:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

if not ENV_MARKER.exists():
    run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'])
    run(PIP + [
        'install', '-q', '--force-reinstall',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    run(PIP + [
        'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy>=1.12,<1.15', 'scikit-learn>=1.6,<1.8',
    ])
    run(PIP + ['install', '-q', '-e', '.[downstream]', 'eotdl', 'tqdm', 'matplotlib', 'ipywidgets'])
    import torch
    cxx11_abi = 'TRUE' if getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', True) else 'FALSE'
    python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    if python_tag != 'cp312':
        raise RuntimeError(f'This prebuilt Mamba preset expects Python 3.12, got {python_tag}.')
    if cxx11_abi != 'TRUE':
        raise RuntimeError(f'This prebuilt Mamba preset expects Torch CXX11 ABI TRUE, got {cxx11_abi}.')
    print('Torch CXX11 ABI:', cxx11_abi)
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    ENV_MARKER.write_text('installed
', encoding='utf-8')
    print('Dependencies installed. Restarting runtime to reload binary modules.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab choose Runtime -> Change runtime type -> GPU.')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)

try:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required for training/reconstruction. Use a fresh GPU Colab runtime, '
        'set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ) from exc

## 4. Drive, dane i checkpoint startowy

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
for directory in [DRIVE_CHECKPOINTS, DRIVE_RECONS, DRIVE_RESULTS, DRIVE_RUNS]:
    directory.mkdir(parents=True, exist_ok=True)


def is_hyperview2_root(path: Path) -> bool:
    required = [
        path / 'train_gt.csv',
        path / 'submission.csv',
        path / 'train/hsi_satellite',
        path / 'test/hsi_satellite',
    ]
    return all(item.exists() for item in required)


def find_hyperview2_root(search_root: Path) -> Path | None:
    if is_hyperview2_root(search_root):
        return search_root
    if search_root.exists():
        for candidate in sorted(search_root.rglob('HYPERVIEW2')):
            if is_hyperview2_root(candidate):
                return candidate
    return None

HV2_ROOT = find_hyperview2_root(DRIVE_HV2_ROOT.parent)
if HV2_ROOT is None:
    raise FileNotFoundError(
        'Nie znaleziono HYPERVIEW2 na Drive. Oczekiwane: '
        f'{DRIVE_HV2_ROOT} z train_gt.csv i katalogami train/test.'
    )

PRETRAINED_CKPT = next((path for path in PRETRAINED_CKPT_CANDIDATES if path.exists()), None)
if PRETRAINED_CKPT is None:
    print('Checked pretrained candidates:')
    for path in PRETRAINED_CKPT_CANDIDATES:
        print(' -', path)
    raise FileNotFoundError('Brakuje checkpointu HySpecNet pretrained na Drive.')

print('HV2_ROOT:', HV2_ROOT)
print('Pretrained:', PRETRAINED_CKPT)
for rel in ['train/hsi_satellite', 'test/hsi_satellite']:
    directory = HV2_ROOT / rel
    print(f'{rel:24s} {len(list(directory.glob("*.npz"))):5d} npz files')
print('Output checkpoints:', DRIVE_CHECKPOINTS)
print('Reconstructions:', DRIVE_RECONS)
print('Results:', DRIVE_RESULTS)

## 5. Split, geometria i rozmiary subsetow

In [ ]:
import numpy as np
from hsi_compression.downstream import build_hyperview2_samples, split_samples
from hsi_compression.downstream.hyperview2 import load_array, load_mask

samples = build_hyperview2_samples(HV2_ROOT, modality='prisma', split='train')
train_samples, val_samples = split_samples(samples, val_fraction=VAL_FRACTION, seed=SEED)
print('Full labeled samples:', len(samples))
print('Train/val split:', len(train_samples), len(val_samples))

for sample in train_samples[:5]:
    cube = load_array(sample.array_path, modality='prisma')
    mask = load_mask(sample.mask_path, shape_hw=tuple(cube.shape[-2:]))
    valid_hw = mask.any(axis=0) if mask.ndim == 3 else mask
    print(sample.sample_id, 'cube=', cube.shape, 'valid_pixels=', int(valid_hw.sum()), '/', valid_hw.size)

fractions = FRACTIONS_BY_PRESET[EVAL_PRESET]
train_subset_counts = {
    fraction: (None if fraction >= 1.0 else max(1, int(round(len(train_samples) * fraction))))
    for fraction in fractions
}
print('Train subset counts:', train_subset_counts)
print('Training val subset:', TRAIN_VAL_SUBSET_BY_PRESET[EVAL_PRESET])
print('Downstream subset:', DOWNSTREAM_TRAIN_SUBSET_BY_PRESET[EVAL_PRESET], DOWNSTREAM_VAL_SUBSET_BY_PRESET[EVAL_PRESET])

## 6. Plan eksperymentow

In [ ]:
import pandas as pd

MODE_SPECS = {
    'scratch': {
        'pretrained': False,
        'trainable_regex': [],
        'description': 'Mamba trained from random initialization on target-domain samples.',
    },
    'pretrained_full': {
        'pretrained': True,
        'trainable_regex': [],
        'description': 'HySpecNet pretrained checkpoint, all parameters fine-tuned.',
    },
    'pretrained_frozen_spectral': {
        'pretrained': True,
        'trainable_regex': ['spatial_condition', 'encoder_to_latent', 'entropy_bottleneck', 'decoder'],
        'description': 'HySpecNet spectral backbone frozen; train spatial/latent/entropy/decoder path.',
    },
}

experiment_rows = []
for fraction in fractions:
    subset_size = train_subset_counts[fraction]
    fraction_tag = f'p{int(round(fraction * 1000)):04d}'
    for mode in RUN_MODES:
        if mode not in MODE_SPECS:
            raise ValueError(f'Unknown mode: {mode}')
        spec = MODE_SPECS[mode]
        experiment_name = f'{EXPERIMENT_PREFIX}_{EVAL_PRESET}_{mode}_{fraction_tag}'
        experiment_rows.append({
            'experiment_name': experiment_name,
            'mode': mode,
            'fraction': fraction,
            'train_subset_size': subset_size,
            'val_subset_size': TRAIN_VAL_SUBSET_BY_PRESET[EVAL_PRESET],
            'epochs': EPOCHS_BY_PRESET[EVAL_PRESET],
            'pretrained': spec['pretrained'],
            'trainable_regex': spec['trainable_regex'],
            'description': spec['description'],
            'checkpoint_best': DRIVE_CHECKPOINTS / f'{experiment_name}_best.pt',
        })

plan_df = pd.DataFrame(experiment_rows)
display(plan_df[['experiment_name', 'mode', 'fraction', 'train_subset_size', 'val_subset_size', 'epochs', 'pretrained', 'trainable_regex']])
plan_path = DRIVE_RESULTS / f'experiment_plan_{EVAL_PRESET}.csv'
plan_df.assign(checkpoint_best=plan_df['checkpoint_best'].astype(str)).to_csv(plan_path, index=False)
print('Saved plan:', plan_path)

## 7. Trening matrixu

In [ ]:
import os
import shutil
import subprocess
import sys
from datetime import datetime

config_path = REPO_DIR / CONFIG_REL
if not config_path.exists():
    raise FileNotFoundError(f'Missing config: {config_path}')

ARTIFACT_CHECKPOINTS = REPO_DIR / 'artifacts/checkpoints'
ARTIFACT_LOGS = REPO_DIR / 'artifacts/logs'
ARTIFACT_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_LOGS.mkdir(parents=True, exist_ok=True)


def run_training(row):
    exp = row['experiment_name']
    drive_best = Path(row['checkpoint_best'])
    if SKIP_COMPLETED_TRAINING and drive_best.exists():
        print(f'[{exp}] checkpoint already exists on Drive, skipping training:', drive_best)
        return

    cmd = [
        sys.executable,
        'scripts/train_hyperview2_compressor.py',
        '--config', str(config_path),
        '--dataset-root', str(HV2_ROOT),
        '--run-name', exp,
        '--override-experiment-name', exp,
        '--override-epochs', str(int(row['epochs'])),
    ]
    if REQUIRE_CUDA:
        cmd.append('--require-cuda')
    if DISABLE_WANDB:
        cmd.append('--disable-wandb')
    if RESUME:
        cmd.append('--resume')
    if row['train_subset_size'] is not None:
        cmd += ['--override-train-subset-size', str(int(row['train_subset_size']))]
    if row['val_subset_size'] is not None:
        cmd += ['--override-val-subset-size', str(int(row['val_subset_size']))]
    if row['pretrained']:
        cmd += ['--pretrained', str(PRETRAINED_CKPT)]
    for pattern in row['trainable_regex']:
        cmd += ['--trainable-regex', pattern]

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('WANDB_CONSOLE', 'wrap')

    log_path = DRIVE_RUNS / f'{exp}_stdout.log'
    print('
=== Training', exp, '===')
    print('Command:', ' '.join(map(str, cmd)))
    print('Log:', log_path)
    if RUN_TRAINING:
        with log_path.open('a', encoding='utf-8', buffering=1) as log_file:
            log_file.write('

=== pretraining transfer run ===
')
            log_file.write(' '.join(map(str, cmd)) + '
')
            process = subprocess.Popen(
                cmd,
                cwd=REPO_DIR,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end='')
                log_file.write(line)
            return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
    else:
        print('RUN_TRAINING=False, command not executed.')

    for suffix in ['best.pt', 'last.pt']:
        src = ARTIFACT_CHECKPOINTS / f'{exp}_{suffix}'
        if src.exists():
            dst = DRIVE_CHECKPOINTS / src.name
            shutil.copy2(src, dst)
            print('Copied checkpoint:', dst)
    for log_src in sorted(ARTIFACT_LOGS.glob(f'{exp}*')):
        if log_src.is_file():
            dst = DRIVE_RUNS / log_src.name
            shutil.copy2(log_src, dst)
            print('Copied summary/log:', dst)


for row in experiment_rows:
    run_training(row)

print('Training matrix cell finished.')

## 8. Rekonstrukcje i downstream quick/full

In [ ]:
from hsi_compression.downstream.hyperview2_compression_eval import (
    CompressionCheckpoint,
    evaluate_downstream_regressors,
    read_reconstruction_summary,
    reconstruct_checkpoint,
    reconstruct_spectral_resample_passthrough,
    save_downstream_artifacts,
)

recon_roots = {}
recon_summaries = {}

resample_variant = 'spectral_resample_passthrough_hyspecnet202_to_230'
resample_root = DRIVE_RECONS / resample_variant / 'HYPERVIEW2'
if resample_root.exists() and not RECOMPUTE_RECONSTRUCTIONS:
    print('Using existing resampling-only reconstruction:', resample_root)
    resample_summary = read_reconstruction_summary(resample_root) or {}
else:
    resample_root, resample_summary = reconstruct_spectral_resample_passthrough(
        source_root=HV2_ROOT,
        recon_parent=DRIVE_RECONS,
        variant_name=resample_variant,
        modality='prisma',
        normalization='reflectance_0_1',
        spectral_mapping_name='hyspecnet_202_approx',
        batch_size=32,
        num_workers=COLAB_DATALOADER_NUM_WORKERS,
        split='train',
    )
recon_roots[resample_variant] = resample_root
recon_summaries[resample_variant] = resample_summary

for row in experiment_rows:
    ckpt_path = Path(row['checkpoint_best'])
    if not ckpt_path.exists():
        print('Skipping missing checkpoint:', ckpt_path)
        continue
    variant = row['experiment_name']
    expected_root = DRIVE_RECONS / variant / 'HYPERVIEW2'
    if expected_root.exists() and not RECOMPUTE_RECONSTRUCTIONS:
        print('Using existing reconstruction:', variant)
        summary = read_reconstruction_summary(expected_root) or {}
        recon_root = expected_root
    elif RUN_RECONSTRUCTION:
        checkpoint = CompressionCheckpoint(
            name=row['experiment_name'],
            path=ckpt_path,
            variant_name=variant,
            modality='prisma',
            compression_normalization='reflectance_0_1',
            recon_feature_normalization='none',
            batch_size=RECON_BATCH_SIZE,
            num_workers=RECON_NUM_WORKERS,
            use_bitstream=True,
            pad_multiple=4,
            min_spatial_size=4,
            allow_in_channel_adapter=False,
            spectral_mapping='hyspecnet_202_approx',
        )
        recon_root, summary = reconstruct_checkpoint(
            checkpoint,
            source_root=HV2_ROOT,
            recon_parent=DRIVE_RECONS,
            split='train',
            device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
            checkpoint_normalization_fallback='reflectance_0_1',
        )
    else:
        print('RUN_RECONSTRUCTION=False and reconstruction does not exist:', expected_root)
        continue
    recon_roots[variant] = recon_root
    recon_summaries[variant] = summary

print('Recon roots:')
for name, root in recon_roots.items():
    print(name, '->', root)

In [ ]:
import json
import torch

results_path = DRIVE_RESULTS / f'compression_downstream_summary_{EVAL_PRESET}.csv'
predictions_path = DRIVE_RESULTS / f'compression_downstream_predictions_{EVAL_PRESET}.csv'
metrics_path = DRIVE_RESULTS / f'compression_downstream_metrics_{EVAL_PRESET}.json'

if RUN_DOWNSTREAM and (RECOMPUTE_REGRESSORS or not results_path.exists()):
    results_df, predictions_df, metrics_payload = evaluate_downstream_regressors(
        hv2_root=HV2_ROOT,
        recon_roots=recon_roots,
        recon_feature_normalizations={name: 'none' for name in recon_roots},
        model_names=REGRESSORS_BY_PRESET[EVAL_PRESET],
        modality='prisma',
        feature_set='mean_std_derivatives',
        original_feature_normalization='reflectance_0_1',
        val_fraction=VAL_FRACTION,
        seed=SEED,
        n_jobs=-1,
        feature_device=torch.device('cuda') if torch.cuda.is_available() else None,
        feature_batch_size=FEATURE_BATCH_SIZE,
        feature_num_workers=FEATURE_NUM_WORKERS,
        max_train_samples=DOWNSTREAM_TRAIN_SUBSET_BY_PRESET[EVAL_PRESET],
        max_val_samples=DOWNSTREAM_VAL_SUBSET_BY_PRESET[EVAL_PRESET],
        verbose=True,
    )
    metrics_payload['reconstruction_summaries'] = recon_summaries
    metrics_payload['experiment_plan'] = plan_df.assign(checkpoint_best=plan_df['checkpoint_best'].astype(str)).to_dict(orient='records')
    paths = save_downstream_artifacts(DRIVE_RESULTS, results_df, predictions_df, metrics_payload)
    # Also save preset-specific aliases so multiple runs can coexist.
    results_df.to_csv(results_path, index=False)
    predictions_df.to_csv(predictions_path, index=False)
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, default=str), encoding='utf-8')
    print('Saved:', paths)
    print('Saved preset aliases:', results_path, predictions_path, metrics_path)
else:
    print('Loading cached downstream results:', results_path)
    results_df = pd.read_csv(results_path)

best = results_df.sort_values('hyperview_score').groupby(['variant', 'mode'], as_index=False).first()
display(best[['variant', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']].sort_values(['mode', 'hyperview_score']))

## 9. Interpretacja: czy pretraining pomaga?

In [ ]:
best_scores = best[['variant', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']].copy()
plan_lookup = plan_df[['experiment_name', 'mode', 'fraction', 'pretrained']].rename(columns={'experiment_name': 'variant', 'mode': 'train_mode'})
best_scores = best_scores.merge(plan_lookup, on='variant', how='left')

main = best_scores[best_scores['mode'] == 'original_train_to_recon_val'].copy()
main = main.sort_values(['fraction', 'hyperview_score'])
print('Main robustness view: original-trained regressor evaluated on reconstructions')
display(main[['fraction', 'train_mode', 'variant', 'model', 'hyperview_score', 'mean_mse', 'mean_mae']])

pivot = main.pivot_table(index='fraction', columns='train_mode', values='hyperview_score', aggfunc='min')
display(pivot)

if {'scratch', 'pretrained_full'}.issubset(set(pivot.columns)):
    pivot['pretrained_full_minus_scratch'] = pivot['pretrained_full'] - pivot['scratch']
if {'scratch', 'pretrained_frozen_spectral'}.issubset(set(pivot.columns)):
    pivot['frozen_spectral_minus_scratch'] = pivot['pretrained_frozen_spectral'] - pivot['scratch']
print('Negative deltas mean pretraining/adaptation beat scratch at the same target-data budget.')
display(pivot)

print('Decision rule:')
print('- If pretrained_full is consistently below scratch at low fractions, HySpecNet pretraining is useful as foundation initialization.')
print('- If frozen_spectral is close to pretrained_full, the learned spectral backbone transfers and only target-domain heads/entropy need adaptation.')
print('- If scratch wins or ties, current pretraining is not useful without a wavelength/sensor-aware adapter.')
print('- Always compare against spectral_resample_passthrough; if Mamba is worse than this control, compression still damages downstream information.')